<a href="https://colab.research.google.com/github/tfxhk/urdu_ocr_codesaviours_si26_Hafiza_Tehreem/blob/main/SI26_Week4%265(urdu)_Tehreem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Required Libraries

In [1]:
!pip install -q transformers sentencepiece accelerate datasets evaluate jiwer pillow pandas torch gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.4 MB/s eta 0:00:00


In [3]:
import torch
from transformers import (
    RobertaTokenizer,
    AutoImageProcessor,
    TrOCRProcessor,
    VisionEncoderDecoderModel
)


In [2]:
import os
import zipfile
from google.colab import files

print("Upload your Urdu OCR zip file:")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_path = "/content/data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("\nDataset extracted successfully!")

Upload your Urdu OCR zip file:


Saving Urdu OCR new zip file.zip to Urdu OCR new zip file.zip

Dataset extracted successfully!


In [5]:
import os
import pandas as pd

# Search for labels CSV file inside /content/data
csv_path = None
for root, dirs, files_list in os.walk("/content/data"):
    for file in files_list:
        if file.startswith("labels") and file.endswith(".csv"):
            csv_path = os.path.join(root, file)
            break

print("Found CSV Path:", csv_path)

# Display sample labels
df = pd.read_csv(csv_path)
print("\nFirst 5 rows of labels:")
print(df.head())

Found CSV Path: /content/data/Urdu OCR/labels.csv.csv

First 5 rows of labels:
                   image                                       text
0          image_01.jfif                                      خواہش
1          image_02.jfif                      نوٹ اپنی زندگی گزاریں
2          image_03.jfif                               خوش رہا کریں
3   quotes/quote_01.jfif  اچھے دنوں کے ليے برے دنوں سے لڑنا پڑتا ہے
4  quotes/quotes_02.jfif                           کی ویکھدے پيے او


# Load Model & Processor

In [6]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

MODEL_NAME = "Khurram123/urdu-poetry-trocr"

print(f"Loading Urdu processor and model ({MODEL_NAME})...")
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Urdu model loaded successfully!")

Using device: cuda
Loading Urdu processor and model (Khurram123/urdu-poetry-trocr)...


processor_config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.28k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/415 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.57M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

Urdu model loaded successfully!


# Auto-Detect CSV & Load Dataset

In [7]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split

SEARCH_ROOT = "/content/data"
csv_path = None

for root, dirs, files_list in os.walk(SEARCH_ROOT):
    for file in files_list:
        if file.startswith("labels") and file.endswith(".csv"):
            csv_path = os.path.join(root, file)
            break
    if csv_path:
        break

if not csv_path:
    raise FileNotFoundError("Could not find any labels CSV file inside /content/data!")

print(f"Found CSV File at: {csv_path}")

class UrduOCRDataset(Dataset):
    def __init__(self, csv_file, search_root, processor):
        raw_df = pd.read_csv(csv_file)
        self.processor = processor
        self.path_map = {}

        for root, _, files in os.walk(search_root):
            for f in files:
                full_path = os.path.join(root, f)
                self.path_map[f] = full_path
                self.path_map[os.path.relpath(full_path, search_root)] = full_path

        valid_rows = []
        for idx, row in raw_df.iterrows():
            img_name = str(row["image"]).strip()
            resolved_path = self.path_map.get(img_name) or self.path_map.get(os.path.basename(img_name))
            if resolved_path and os.path.exists(resolved_path):
                valid_rows.append({"image_path": resolved_path, "text": str(row["text"]).strip()})

        self.data = pd.DataFrame(valid_rows)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = row["image_path"]

        image = Image.open(image_path).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            truncation=True,
            max_length=128
        ).input_ids

        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(labels),
            "raw_text": row["text"]
        }

dataset = UrduOCRDataset(
    csv_file=csv_path,
    search_root=SEARCH_ROOT,
    processor=processor
)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

print(f"Dataset Ready! Total: {len(dataset)} | Train: {len(train_dataset)} | Test: {len(test_dataset)}")

Found CSV File at: /content/data/Urdu OCR/labels.csv.csv
Dataset Ready! Total: 8 | Train: 6 | Test: 2


# Training Loop with Train Loss Tracking

In [8]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)
num_epochs = 10

print("--- Starting Model Fine-Tuning ---\n")
for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0.0

    for batch in train_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Epoch {epoch + 1:02d}/{num_epochs:02d} | Train Loss: {avg_train_loss:.4f}")

print("\nTraining complete!")

--- Starting Model Fine-Tuning ---

Epoch 01/10 | Train Loss: 7.6789
Epoch 02/10 | Train Loss: 5.7836
Epoch 03/10 | Train Loss: 4.8885
Epoch 04/10 | Train Loss: 4.3325
Epoch 05/10 | Train Loss: 3.8884
Epoch 06/10 | Train Loss: 3.6223
Epoch 07/10 | Train Loss: 3.1541
Epoch 08/10 | Train Loss: 3.0028
Epoch 09/10 | Train Loss: 2.4364
Epoch 10/10 | Train Loss: 2.4510

Training complete!


# Full Evaluation (Test Loss, Accuracy, Metrics, & Text Preview)

In [10]:
# Install evaluation dependencies for current runtime session
!pip install -q evaluate jiwer

import evaluate

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

model.eval()

total_test_loss = 0.0
correct_exact_matches = 0
total_samples = 0

predictions = []
references = []

print("--- Evaluating Model on Test Set ---\n")

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        # 1. Compute Test Loss
        outputs = model(pixel_values=pixel_values, labels=labels)
        total_test_loss += outputs.loss.item()

        # 2. Generate Predicted Text Sequence
        generated_ids = model.generate(
            pixel_values,
            max_length=256,
            num_beams=5,
            repetition_penalty=3.0,
            length_penalty=2.0,
            early_stopping=False
        )

        pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # 3. Decode Ground Truth
        clean_labels = labels.clone()
        clean_labels[clean_labels == -100] = processor.tokenizer.pad_token_id
        ref_text = processor.batch_decode(clean_labels, skip_special_tokens=True)

        # Collect metrics data
        for p, r in zip(pred_text, ref_text):
            p_clean = p.strip()
            r_clean = r.strip()

            predictions.append(p_clean)
            references.append(r_clean)

            total_samples += 1
            if p_clean == r_clean:
                correct_exact_matches += 1

# Calculate Overall Summary Metrics
avg_test_loss = total_test_loss / len(test_loader) if len(test_loader) > 0 else 0.0
cer = cer_metric.compute(predictions=predictions, references=references)
wer = wer_metric.compute(predictions=predictions, references=references)
exact_accuracy = (correct_exact_matches / total_samples) * 100 if total_samples > 0 else 0.0

print("=" * 50)
print("             EVALUATION SUMMARY RESULTS           ")
print("=" * 50)
print(f"Average Test Loss:          {avg_test_loss:.4f}")
print(f"Character Error Rate (CER): {cer:.4f}")
print(f"Word Error Rate (WER):      {wer:.4f}")
print(f"Exact Match Accuracy:       {exact_accuracy:.2f}% ({correct_exact_matches}/{total_samples})")
print("=" * 50)

# Display Text Before Launching Gradio
print("\n--- SAMPLE PREDICTIONS VS ACTUAL TEXT ---")
for idx, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"\nSample {idx + 1}:")
    print(f"  Predicted Text: '{pred}'")
    print(f"  Actual Text:    '{ref}'")
    print("-" * 40)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 112.5 MB/s eta 0:00:00


--- Evaluating Model on Test Set ---

             EVALUATION SUMMARY RESULTS           
Average Test Loss:          3.2103
Character Error Rate (CER): 0.8421
Word Error Rate (WER):      1.0000
Exact Match Accuracy:       0.00% (0/2)

--- SAMPLE PREDICTIONS VS ACTUAL TEXT ---

Sample 1:
  Predicted Text: 'نایروزے'
  Actual Text:    'آپ جا سکتے ہیں'
----------------------------------------

Sample 2:
  Predicted Text: 'نایخہوزدکرشگ'
  Actual Text:    'چاند کسی کا ہو نہیں سکتا'
----------------------------------------


# Save model

In [12]:
output_dir = "./urdu_trocr_model"
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

print(f"\nModel and processor successfully saved locally to '{output_dir}'")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model and processor successfully saved locally to './urdu_trocr_model'


In [13]:
!pip install -q evaluate jiwer urduhack

import torch
import evaluate
import re

# Load evaluation metrics
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

# Function to normalize Urdu Unicode differences and remove diacritics
def normalize_urdu(text):
    if not isinstance(text, str):
        return ""

    # Remove diacritics (Zabar, Zair, Paesh, Tanween, etc.)
    text = re.sub(r'[\u064B-\u0652\u0670]', '', text)

    # Normalize character variants (Arabic -> Urdu equivalents)
    text = text.replace('ك', 'ک')
    text = text.replace('ي', 'ی').replace('ى', 'ی').replace('ئ', 'ی')
    text = text.replace('ہ', 'ھ') # Standardize He variants
    text = re.sub(r'\s+', ' ', text).strip() # Standardize multiple spaces

    return text

model.eval()
total_test_loss = 0.0
correct = 0
total = 0

predictions = []
references = []
norm_predictions = []
norm_references = []

print("--- Running Normalized Evaluation ---\n")

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        # Compute Loss
        outputs = model(pixel_values=pixel_values, labels=labels)
        total_test_loss += outputs.loss.item()

        # Generate predicted text
        generated_ids = model.generate(
            pixel_values,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

        pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # Decode ground truth
        labels[labels == -100] = processor.tokenizer.pad_token_id
        ref_text = processor.batch_decode(labels, skip_special_tokens=True)

        for p, r in zip(pred_text, ref_text):
            predictions.append(p.strip())
            references.append(r.strip())

            # Apply Unicode Normalization
            p_norm = normalize_urdu(p)
            r_norm = normalize_urdu(r)

            norm_predictions.append(p_norm)
            norm_references.append(r_norm)

            total += 1
            if p_norm == r_norm:
                correct += 1

# Calculate Metrics
avg_loss = total_test_loss / len(test_loader) if len(test_loader) > 0 else 0
accuracy = (correct / total) * 100 if total > 0 else 0
cer = cer_metric.compute(predictions=norm_predictions, references=norm_references)
wer = wer_metric.compute(predictions=norm_predictions, references=norm_references)

print("=" * 55)
print(f" Average Test Loss:           {avg_loss:.4f}")
print(f" Character Error Rate (CER):  {cer:.4f}  (Lower is better)")
print(f" Word Error Rate (WER):       {wer:.4f}  (Lower is better)")
print(f" Normalized Accuracy:         {accuracy:.2f}% ({correct}/{total} correct)")
print("=" * 55)

print("\n--- DETAILED COMPARISON ---")
for idx, (raw_p, raw_r, norm_p, norm_r) in enumerate(zip(predictions, references, norm_predictions, norm_references)):
    print(f"\nSample {idx+1}:")
    print(f"  Raw Predicted:  '{raw_p}'")
    print(f"  Raw GroundTruth:'{raw_r}'")
    print(f"  Norm Predicted: '{norm_p}'")
    print(f"  Norm GroundTruth:'{norm_r}'")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.8/679.8 kB 50.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 4.1 MB/s eta 0:00:00
--- Running Normalized Evaluation ---

 Average Test Loss:           3.2103
 Character Error Rate (CER):  3.1579  (Lower is better)
 Word Error Rate (WER):       1.0000  (Lower is better)
 Normalized Accuracy:         0.00% (0/2 correct)

--- DETAILED COMPARISON ---

Sample 1:
  Raw Predicted:  'نننننننننننننننننننننننننننننننننننننننننننننننننننننننننننننن'
  Raw GroundTruth:'آپ جا سکتے ہیں'
  Norm Predicted: 'نننننننننننننننننننننننننننننننننننننننننننننننننننننننننننننن'
  Norm GroundTruth:'آپ جا سکتے ھیں'

Sample 2:
  Raw Predicted:  'نینینیینینینینینینینینینینینینینینینینینینینینینینینیننینینینی'
  Raw GroundTruth:'چاند کسی کا ہو نہیں سکتا'
  Norm Predicted: 'نینینیینینینینینینینینینینینینینینینینینینینینی